# 10 — Reservar vídeos
Cria a execução e reserva os vídeos pendentes para as próximas tasks do Workflow.

In [ ]:
import json

from youtube_etl_genai.main import _get_spark_session
from youtube_etl_genai.observability import TaskExecution, configure_job_logging
from youtube_etl_genai.pipeline import claim_targets_step

TASK_KEY = "claim_targets"
configure_job_logging()

dbutils.widgets.text("batch_size", "20")
dbutils.widgets.text("catalog", "youtube_lakehouse")
dbutils.widgets.text("task_run_id", "")

spark = _get_spark_session()
catalog = dbutils.widgets.get("catalog")
with TaskExecution(
    spark=spark,
    catalog=catalog,
    task_key=TASK_KEY,
    task_run_id=dbutils.widgets.get("task_run_id") or None,
) as task_execution:
    result = claim_targets_step(
        spark=spark,
        batch_size=dbutils.widgets.get("batch_size"),
        catalog=catalog,
    )
    task_execution.complete_from_result(result)
dbutils.jobs.taskValues.set(key="ingestion_id", value=result["ingestion_id"])
print(json.dumps(result, sort_keys=True))
